# DEMO 3: (WorkflowAware) multiple workflow workloads

In this demo how our `WorkflowAware` scheduler implementation can reduce total execution time of the workload when it consists of multiple and diverse set of workflows, emulating real-life computational workloads.

In [ ]:
import os
import platform
import subprocess
from enum import Enum

import numpy as np
import pandas as pd


## Step 1: Generating Data

We createa a workload consisting of multiple subsequently submitted workflow with diverse topologies below.
We use a list of seed workflows which we randomly select and add to the total workload with increasing submittion and deadline timestamps.

(The data for this demo is already generated, so no need to rerun it)

In [2]:
seeds = [
    "../input/synthetic_traces/balanced_tree_depth2_branching2_deadline_default",
    "../input/synthetic_traces/balanced_tree_depth4_branching2_deadline_default",
    "../input/synthetic_traces/highly_dependent_length10_deadline_default",
    "../input/synthetic_traces/mostly_parallel_n30_deadline_default",
    "../input/synthetic_traces/random_dag_n25_edge_prob0.05_seed1_deadline_default",
    "../input/synthetic_traces/real_world_cybershake_n50_deadline_default",
    "../input/synthetic_traces/real_world_genome_n50_deadline_default",
    "../input/synthetic_traces/real_world_sipht_n50_deadline_default",
]

In [3]:
final_tasks = []
final_fragments = []

In [4]:
rng = np.random.default_rng(seed=42)

In [5]:
submission_time = int(
    pd.to_datetime("2022-07-01 12:00:00", utc=True).timestamp() * 1000
)

for i in range(0, 100):
    seed = rng.choice(seeds)
    tasks = pd.read_parquet(seed + "/tasks.parquet")
    fragments = pd.read_parquet(seed + "/fragments.parquet")

    # ids first
    tasks["id"] += i * 100
    tasks["parents"] += i * 100
    tasks["children"] += i * 100

    fragments["id"] += i * 100

    # submition time
    n_second_offset = (
        rng.integers(1, 10) * 1000
    )  # add a nother task in either 1 or 10 seconds
    submission_time += n_second_offset
    tasks["submission_time"] = submission_time
    tasks["deadline"] = tasks["submission_time"] + 100

    final_tasks.append(tasks.copy())
    final_fragments.append(fragments.copy())

In [ ]:
final_tasks_df = pd.concat(final_tasks).reset_index(drop=True)
final_fragments_df = pd.concat(final_fragments).reset_index(drop=True)

# final_tasks_df.to_parquet("demo_workloads/demo_3/tasks.parquet")
# final_fragments_df.to_parquet("demo_workloads/demo_3/fragments.parquet")

In [11]:
final_tasks_df.sample(10)

,id,submission_time,duration,cpu_count,cpu_capacity,mem_capacity,parents,children,deferrable,deadline
1932,5724,1656677110000,96974,1,1323.219422,1353,[5708],[],True,1656677110100
188,637,1656676843000,1210,1,1000.000000,500,[614],[649],True,1656676843100
3064,9404,1656677270000,30000,1,500.000000,200,[9401],"[9409, 9410]",True,1656677270100
1992,6021,1656677121000,60000,1,1000.000000,500,[],[6030],True,1656677121100
344,1021,1656676862000,74274,1,1404.126314,1624,"[1000, 1005, 1018]",[],True,1656676862100
2855,8834,1656677244000,1440,1,1000.000000,500,[8811],[8849],True,1656677244100
2772,8701,1656677242000,152250,1,1000.000000,500,[],"[8712, 8713, 8714, 8715, 8716, 8717, 8718, 871...",True,1656677242100
1280,4016,1656677017000,990,1,1000.000000,500,[],[4034],True,1656677017100
1602,4829,1656677065000,30000,1,500.000000,200,[4814],[],True,1656677065100
2888,8917,1656677249000,1240,1,1000.000000,500,[],[8934],True,1656677249100


## Step 2: Running the workload

Now, we can simply run OpenDC and observe the results!

In [12]:
class SchedulerEnum(Enum):
    DEFAULT_SCHEDULER = 0
    WORKFLOW_AWARE_SCHEDULER = 1

In [13]:
# DETECT JAVA HOME

os_type = platform.system()

if os_type == "Darwin":  # macOS
    os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@21"
    os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]
elif os_type == "Linux":  # Ubuntu/Linux
    java_home = os.environ.get("JAVA_HOME", "/usr/lib/jvm/java-21-openjdk-amd64")
    os.environ["JAVA_HOME"] = java_home
    os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]
else:
    print(
        f"Warning: Unsupported OS ({os_type}). JAVA_HOME may need to be set manually."
    )


In [14]:
experiment = "demo_experiments/demo_experiment_3.json"

subprocess.run(
    [
        "../OpenDCExperimentRunner/bin/OpenDCExperimentRunner",
        "--experiment-path",
        experiment,
    ]
)



 Running scenario: 0 
 Starting seed: 0 


Simulating...   0% [                                       ] 0/2 (0:00:00 / ?) 

Creating ComputeScheduler: Random with 10 hosts
09:24:37.965 [WARN ] org.opendc.compute.simulator.telemetry.ComputeMetricReader - 
					Metrics after 24 hours:
						Tasks Total: 3234
						Tasks Active: 20
						Tasks Pending: 153
						Tasks Completed: 2939
						Tasks Terminated: 0



 Running scenario: 1 
 Starting seed: 0 
Creating ComputeScheduler: WorkflowAware with 10 hosts


Simulating...  50% [================                 ] 1/2 (0:00:01 / 0:00:01) 

09:24:38.659 [WARN ] org.opendc.compute.simulator.telemetry.ComputeMetricReader - 
					Metrics after 24 hours:
						Tasks Total: 3234
						Tasks Active: 20
						Tasks Pending: 1060
						Tasks Completed: 806
						Tasks Terminated: 0



Simulating... 100% [=================================] 2/2 (0:00:01 / 0:00:00) 


CompletedProcess(args=['../OpenDCExperimentRunner/bin/OpenDCExperimentRunner', '--experiment-path', 'demo_experiments/demo_experiment_3.json'], returncode=0)

In [15]:
output_path = "output/demo_experiment/raw-output/{}/seed=0/"
for scheduler_type in SchedulerEnum:
    df_host = pd.read_parquet(output_path.format(scheduler_type.value) + "host.parquet")
    df_powerSource = pd.read_parquet(
        output_path.format(scheduler_type.value) + "powerSource.parquet"
    )
    df_task = pd.read_parquet(output_path.format(scheduler_type.value) + "task.parquet")
    df_service = pd.read_parquet(
        output_path.format(scheduler_type.value) + "service.parquet"
    )

    # calculating metrics:
    runtime = pd.to_timedelta(
        df_service.timestamp.max() - df_service.timestamp.min(), unit="ms"
    )

    energy = df_powerSource.energy_usage.sum() / 3_600_000  # convert energy to kWh
    carbon = df_powerSource.carbon_emission.sum() / 1000  # convert carbon to kg

    print(f"Scheduler: {scheduler_type.name}")
    print(f"The workload was finished in {runtime}")
    print(f"The datacenter used {energy:.2f} kWh during the workload")
    print(f"The datacenter emitted {carbon:.2f} kg during the workload")

    print("=" * 50 + "\n")

Scheduler: DEFAULT_SCHEDULER
The workload was finished in 1 days 11:50:40.580000
The datacenter used 11.80 kWh during the workload
The datacenter emitted 3.56 kg during the workload

Scheduler: WORKFLOW_AWARE_SCHEDULER
The workload was finished in 1 days 10:11:49.580000
The datacenter used 11.28 kWh during the workload
The datacenter emitted 3.32 kg during the workload



We can see that `WorkflowAware` scheduler already shortens the execution total execution time by a few hours! This in turn also produces less carbon emmisions!